11_SOTA_and_diagnostics.py
Journal-revision experiments (all numbers are real, computed here):

  (1) SOTA baselines: LightGBM and CatBoost trained with the SAME leakage-safe
      pipeline and the SAME 5-fold split as the in-house XGBoost, so the
      comparison in the paper is fair.
  (2) Failure-case analysis: the development-set listings with the largest blend
      residuals, with their key characteristics.
  (3) Feature-family inventory from the engineered matrix.

Writes outputs/sota_results.txt and outputs/failure_cases.txt


In [19]:
# Notebook compatibility helper
import os
os.environ['PYTHONWARNINGS'] = 'ignore'  # also silences warnings from n_jobs=-1 joblib subprocesses
from pathlib import Path
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

In [20]:
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
OUT = Path('outputs')
RS, K = 42, 5
train = pd.read_parquet(OUT / 'train_local.parquet')
TARGET, ID = 'blocked_days_Q1_2026', 'id'
y = train[TARGET].astype(float).values
X = train.drop(columns=[TARGET, ID]).reset_index(drop=True)

cat_all = X.select_dtypes(exclude='number').columns.tolist()
card = {c: X[c].nunique(dropna=False) for c in cat_all}
cat_high = [c for c, n in card.items() if n > 15]
cat_low = [c for c, n in card.items() if n <= 15]
num_cols = X.select_dtypes(include='number').columns.tolist()

In [21]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state

    def _smap(self, x, yy):
        st = pd.DataFrame({'c': x, 'y': yy}).groupby('c')['y'].agg(['mean', 'count'])
        return ((st['count'] * st['mean'] + self.smoothing * self.gm_)
                / (st['count'] + self.smoothing)).to_dict()

    def fit(self, X, y):
        y = np.asarray(y, float); self.gm_ = float(y.mean())
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return self

    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = (X[c].astype(str).fillna('__nan__').map(self.maps_[c])
                       .fillna(self.gm_).astype('float32'))
        return Xo

    def fit_transform(self, X, y=None, **kw):
        y = np.asarray(y, float); self.gm_ = float(y.mean())
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = np.full(len(X), self.gm_, dtype='float32')
        kf = KFold(self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(X):
            for c in self.cols:
                m = self._smap(X[c].astype(str).fillna('__nan__').iloc[tr], y[tr])
                Xo.iloc[va, Xo.columns.get_loc(c)] = (
                    X[c].astype(str).fillna('__nan__').iloc[va].map(m)
                       .fillna(self.gm_).astype('float32').values)
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return Xo


In [22]:
def make_pp():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
        ('low', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_low),
        ('high', Pipeline([('te', KFoldTargetEncoder(cat_high, 5, 20, RS))]), cat_high),
    ])


## Hyperparameter Tuning for LightGBM & CatBoost

Previously both models used **fixed, untuned** hyperparameters. This adds the same kind of
fold-level-early-stopping random search already used for XGBoost in `08_XGBoost_v2.ipynb` /
`09_XGBoost_v3.ipynb`: for each candidate config, run the identical 5-fold CV split, carve a
10% inner-validation slice inside each fold's training portion, fit with early stopping
watching that slice, and average the resulting fold MSEs. The winning config per model family
is then reused — still with early stopping — for the final OOF / local-test predictions below,
so the reported iteration count actually matches what the search selected.

In [23]:
import lightgbm as lgb

def evaluate_gbm(model_name, ctor, params, return_oof=False, return_models=False):
    """5-fold CV with a 10% inner-validation slice per fold for early stopping."""
    warnings.filterwarnings('ignore')  # re-assert: Jupyter can reset filters between cells,
    # and this loop otherwise prints a SimpleImputer warning every fold for the 3 all-NaN
    # columns (estimated_revenue_l365d, price_implied, implied_nightly_rate) -- harmless,
    # median imputation just leaves those columns at 0/NaN, but very noisy at 200 calls.
    kf = KFold(K, shuffle=True, random_state=RS)
    fold_mses = []
    oof = np.zeros(len(y)) if return_oof else None
    models_out = []
    for fold, (tr, va) in enumerate(kf.split(X)):
        pp = make_pp()
        Xtr_full = pp.fit_transform(X.iloc[tr], y[tr])
        Xva = pp.transform(X.iloc[va])
        Xtr, Xin, ytr, yin = train_test_split(Xtr_full, y[tr], test_size=0.10, random_state=RS + fold)
        m = ctor(params, RS + fold)
        if model_name == 'LightGBM':
            m.fit(Xtr, ytr, eval_set=[(Xin, yin)], callbacks=[lgb.early_stopping(50, verbose=False)])
        else:  # CatBoost
            m.fit(Xtr, ytr, eval_set=(Xin, yin), use_best_model=True, verbose=False)
        pred = np.clip(m.predict(Xva), 0, 90)
        fold_mses.append(mean_squared_error(y[va], pred))
        if return_oof: oof[va] = pred
        if return_models: models_out.append((m, pp))
    return {'mse_mean': float(np.mean(fold_mses)), 'mse_std': float(np.std(fold_mses)),
            'oof': oof, 'models': models_out}


# Independent seed from the XGBoost notebooks' search RNGs (RANDOM_STATE and RANDOM_STATE+1000)
rng = np.random.default_rng(RS + 2000)

def sample_lgb():
    return dict(
        n_estimators=3000,
        learning_rate=float(np.exp(rng.uniform(np.log(0.02), np.log(0.10)))),
        num_leaves=int(rng.integers(20, 120)),
        min_child_samples=int(rng.integers(5, 60)),
        subsample=float(rng.uniform(0.65, 0.95)),
        colsample_bytree=float(rng.uniform(0.65, 0.95)),
        reg_lambda=float(np.exp(rng.uniform(np.log(0.5), np.log(8.0)))),
    )

def sample_cat():
    # No `subsample` here: CatBoost's default Bayesian bootstrap doesn't accept it
    # (would need bootstrap_type='Bernoulli' to enable subsample -- kept simple/robust here).
    return dict(
        iterations=3000,
        learning_rate=float(np.exp(rng.uniform(np.log(0.02), np.log(0.10)))),
        depth=int(rng.integers(4, 9)),
        l2_leaf_reg=float(np.exp(rng.uniform(np.log(0.5), np.log(8.0)))),
    )

N_ITER_GBM = 40  # raised from 20 -- widen the LightGBM/CatBoost search budget
search_results = {}
for model_name, sampler, ctor in [
    ('LightGBM', sample_lgb, lambda p, seed: LGBMRegressor(**p, random_state=seed, n_jobs=-1, verbose=-1)),
    ('CatBoost', sample_cat, lambda p, seed: CatBoostRegressor(**p, random_seed=seed, verbose=0, early_stopping_rounds=50)),
]:
    print(f'\n>>> Tuning {model_name} ({N_ITER_GBM} configs)...')
    best_mse, best_cfg, rows = np.inf, None, []
    t0 = time.time()
    for i in range(N_ITER_GBM):
        cfg = sampler()
        t1 = time.time()
        res = evaluate_gbm(model_name, ctor, cfg)
        elapsed = time.time() - t1
        rows.append({**cfg, 'mse_mean': res['mse_mean'], 'mse_std': res['mse_std'], 'seconds': elapsed})
        flag = ''
        if res['mse_mean'] < best_mse:
            best_mse, best_cfg = res['mse_mean'], cfg
            flag = '  <- NEW BEST'
        print(f"[{model_name} {i+1:2d}/{N_ITER_GBM}] MSE={res['mse_mean']:7.3f} +- {res['mse_std']:5.3f} | {elapsed:5.1f}s{flag}")
    print(f'{model_name} search done in {(time.time()-t0)/60:.1f} min | best MSE={best_mse:.3f}')
    print('best config:', best_cfg)
    search_results[model_name] = {'best_cfg': best_cfg, 'best_mse': best_mse, 'ctor': ctor}
    pd.DataFrame(rows).sort_values('mse_mean').to_csv(OUT / f'{model_name.lower()}_search_log.csv', index=False)
    with open(OUT / f'{model_name.lower()}_best_params.json', 'w') as f:
        json.dump(best_cfg, f, indent=2)


>>> Tuning LightGBM (40 configs)...
[LightGBM  1/40] MSE=298.177 +- 3.472 |  48.1s  <- NEW BEST
[LightGBM  2/40] MSE=301.079 +- 1.702 |  26.6s
[LightGBM  3/40] MSE=297.860 +- 2.843 |  63.1s  <- NEW BEST
[LightGBM  4/40] MSE=302.111 +- 1.445 |  28.4s
[LightGBM  5/40] MSE=296.494 +- 2.201 |  77.9s  <- NEW BEST
[LightGBM  6/40] MSE=298.694 +- 3.729 |  66.7s
[LightGBM  7/40] MSE=302.125 +- 3.285 |  24.7s
[LightGBM  8/40] MSE=299.332 +- 3.486 | 145.4s
[LightGBM  9/40] MSE=300.261 +- 3.331 |  42.8s
[LightGBM 10/40] MSE=300.571 +- 2.676 |  29.8s
[LightGBM 11/40] MSE=297.483 +- 2.473 |  51.8s
[LightGBM 12/40] MSE=302.035 +- 2.266 |  19.8s
[LightGBM 13/40] MSE=299.025 +- 3.661 |  33.9s
[LightGBM 14/40] MSE=299.332 +- 4.278 |  22.9s
[LightGBM 15/40] MSE=302.777 +- 3.011 |  17.8s
[LightGBM 16/40] MSE=300.388 +- 2.612 |  19.5s
[LightGBM 17/40] MSE=301.432 +- 3.255 |  23.1s
[LightGBM 18/40] MSE=297.781 +- 1.635 |  31.4s
[LightGBM 19/40] MSE=302.175 +- 1.288 |  11.4s
[LightGBM 20/40] MSE=299.696 +-

In [24]:
# Load local test set
test_local_df = pd.read_parquet(OUT / 'test_local.parquet')
X_local_test = test_local_df.drop(columns=[TARGET, ID]).reindex(columns=X.columns).reset_index(drop=True)
y_local_test = test_local_df[TARGET].astype(float).values

lines = ['SOTA BASELINES (5-fold CV, same split as XGBoost, TUNED)',
         '=' * 70, f'{"Model":<12}{"MSE":>10}{"MAE":>8}{"R2":>8}{"MSE_std":>10}{"sec":>8}']
for model_name, info in search_results.items():
    best_cfg, ctor = info['best_cfg'], info['ctor']
    t0 = time.time()
    final = evaluate_gbm(model_name, ctor, best_cfg, return_oof=True, return_models=True)
    sec = time.time() - t0
    oof = final['oof']
    mse, mae, r2 = mean_squared_error(y, oof), mean_absolute_error(y, oof), r2_score(y, oof)
    lines.append(f'{model_name:<12}{mse:>10.2f}{mae:>8.2f}{r2:>8.3f}{final["mse_std"]:>10.2f}{sec:>8.1f}')
    print(lines[-1])
    np.save(OUT / f'oof_{model_name}.npy', oof)

    # Local test prediction: average across the 5 fold models (each already early-stopped)
    test_local_pred = np.zeros(len(X_local_test))
    for m, pp in final['models']:
        Xtl = pp.transform(X_local_test)
        test_local_pred += np.clip(m.predict(Xtl), 0, 90)
    test_local_pred /= len(final['models'])

    local_test_mse = mean_squared_error(y_local_test, test_local_pred)
    np.save(OUT / f'test_local_pred_{model_name}.npy', test_local_pred)
    print(f'  Local Test MSE: {local_test_mse:.3f} | Saved test_local_pred_{model_name}.npy')

rep = '\n'.join(lines)
(OUT / 'sota_results.txt').write_text(rep, encoding='utf-8')
print('\nsaved sota_results.txt')

LightGBM        296.49   10.20   0.529      2.20    36.5
  Local Test MSE: 293.103 | Saved test_local_pred_LightGBM.npy
CatBoost        301.45   10.43   0.522      3.54    35.8
  Local Test MSE: 299.182 | Saved test_local_pred_CatBoost.npy

saved sota_results.txt
